# Modelado de abandono en RetailNova

Este notebook desarrolla un flujo supervisado y no supervisado para priorizar clientes con riesgo de abandono. La selección de transformaciones, algoritmos y umbrales ocurre solo con entrenamiento y validación interna; el conjunto test queda reservado para la evaluación final. Los datos y resultados son sintéticos y no representan desempeño comercial real.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("No se encontró src/paths.py en esta carpeta ni en sus directorios padre.")
sys.path.insert(0, str(ROOT / "src"))
from modeling import (
    compare_balance_methods,
    evaluate_supervised_models,
    evaluate_clustering,
    evaluate_final_test,
    load_test_data,
    load_training_data,
    optimize_models,
    prepare_validation,
    select_model,
)


## 1. Carga de datos y protocolo de evaluación

**Qué haremos:** cargaremos los splits guardados por el EDA, construiremos la variable de tasa de devoluciones y separaremos una validación estratificada dentro de train.

**Por qué importa en RetailNova:** la validación interna permite comparar estrategias sin usar clientes del holdout; las columnas que describen el período futuro y los identificadores no entrarán como predictores.

In [2]:
train_df = load_training_data()
prepared = prepare_validation(train_df)
print(f"Train raw: {len(train_df):,} clientes")
print(f"Validación interna: {len(prepared['y_valid']):,} clientes")
print(f"Prevalencia de abandono en train: {train_df['churn_6m'].mean():.2%}")
print("El conjunto test no se ha cargado y permanece reservado.")

Train raw: 148,000 clientes
Validación interna: 29,600 clientes
Prevalencia de abandono en train: 23.94%
El conjunto test no se ha cargado y permanece reservado.


### Conclusión de la sección 1

Se cargaron 148.000 clientes de train y se separaron 118.400 para ajuste y 29.600 para validación interna. El holdout de 37.000 clientes no se carga en esta sección y queda reservado para la evaluación final.

## 2. Balanceo de clases sin sobremuestreo por defecto

**Qué haremos:** compararemos regresión logística sin balanceo, pesos de clase, Random Forest con pesos y Balanced Random Forest usando la misma validación y Average Precision, además de precision, recall y tiempo.

**Por qué importa en RetailNova:** la clase de abandono es minoritaria; una estrategia que optimiza recuperación de abandonos debe contrastarse con el costo operativo y evitar duplicar ejemplos sin evidencia de que sea necesario.

In [3]:
balance_results = compare_balance_methods(prepared)
display(balance_results)
best_balance = balance_results.iloc[0]
print(
   f"Mejor Average Precision: {best_balance['metodo']} "
   f"({best_balance['average_precision']:.4f}); "
   f"tiempo: {best_balance['tiempo_segundos']:.1f} s."
)

                         metodo  ...  tiempo_segundos
0  Logística con pesos de clase  ...         0.521944
1        Logística sin balanceo  ...         0.670384
2      Random Forest balanceado  ...        17.050613
3       Random Forest con pesos  ...        19.880146

[4 rows x 5 columns]
Mejor Average Precision: Logística con pesos de clase (0.4103); tiempo: 0.5 s.


### Conclusión de la sección 2

La logística con pesos de clase obtuvo el mayor Average Precision (0,4103), muy similar al modelo sin balanceo (0,4103), pero elevó el recall a umbral 0,5 de 0,101 a 0,592. Para este conjunto sintético se prefiere el peso de clase por esa recuperación adicional sin sobremuestreo; el costo de falsos positivos debe valorarse con negocio.

## 3. Segmentación no supervisada

**Qué haremos:** evaluaremos K-Means, Gaussian Mixture y clustering aglomerativo en una muestra reproducible de train; compararemos silhouette, Davies–Bouldin y Calinski–Harabasz. Para K-Means mostraremos la curva del codo y visualizaremos la segmentación en dos componentes PCA.

**Por qué importa en RetailNova:** los segmentos pueden ayudar a describir perfiles de compra y orientar KPI o campañas, pero no sustituyen la clasificación de abandono. El objetivo no se utiliza para formar los grupos.

In [4]:
clustering_results, clustering_winner, kmeans_k = evaluate_clustering(train_df)
display(clustering_results.head(10))
print(f"Algoritmo de clustering conservado: {clustering_winner}")
print(f"Configuración K-Means mostrada en PCA: k={kmeans_k}")
print("Gráfico guardado en outputs/graphs/clustering_codo_pca.png")
print("Artefacto guardado en outputs/model/modelo_clustering_retailnova.joblib")

          algoritmo  k  ...  calinski_harabasz       inercia
0            KMeans  3  ...         295.492550  13099.195856
1            KMeans  2  ...         330.947102  14304.375324
2  Gaussian Mixture  2  ...         330.947102           NaN
3     Agglomerative  3  ...         226.356426           NaN
4     Agglomerative  2  ...         251.908518           NaN
5     Agglomerative  4  ...         208.395912           NaN
6            KMeans  4  ...         262.454725  12314.784468
7            KMeans  5  ...         242.982406  11658.074156
8            KMeans  6  ...         215.386282  11314.001446
9  Gaussian Mixture  4  ...         134.148230           NaN

[10 rows x 6 columns]
Algoritmo de clustering conservado: KMeans k=3
Configuración K-Means mostrada en PCA: k=3
Gráfico guardado en outputs/graphs/clustering_codo_pca.png
Artefacto guardado en outputs/model/modelo_clustering_retailnova.joblib


### Conclusión de la sección 3

K-Means con k=3 fue la mejor configuración por silhouette (0,135) entre las técnicas evaluadas; su distribución se muestra en PCA junto con el codo de K-Means. La separación es moderada y el Davies–Bouldin complementa la comparación. Los grupos son exploratorios y requieren validación con datos reales antes de tratarlos como segmentos comerciales.

## 4. Búsqueda de hiperparámetros con Optuna

**Qué haremos:** optimizaremos regresión logística, Random Forest, Extra Trees y Balanced Random Forest con TPE. Cada prueba se ajusta con el subgrupo de entrenamiento interno y se puntúa en la validación por Average Precision.

**Por qué importa en RetailNova:** explorar hiperparámetros con una métrica sensible a clases desbalanceadas mejora la comparación frente a configuraciones por defecto. El número de pruebas se mantiene acotado para que el experimento sea reproducible.

In [5]:
optuna_results = optimize_models(prepared, n_trials=5)
display(optuna_results)
print("Historial comparativo guardado en outputs/graphs/optuna_comparacion.png")

              algoritmo  ...  n_trials
0    LogisticRegression  ...         5
1            ExtraTrees  ...         5
2  BalancedRandomForest  ...         5
3          RandomForest  ...         5

[4 rows x 4 columns]
Historial comparativo guardado en outputs/graphs/optuna_comparacion.png


### Conclusión de la sección 4

Optuna encontró una regresión logística con pesos de clase como mejor configuración (Average Precision de validación: 0,4103). Se guardaron los mejores parámetros de las cuatro familias; la comparación detallada a umbral común continúa en la sección siguiente. Test sigue cerrado.

## 5. Evaluación de modelos supervisados

**Qué haremos:** evaluaremos en la misma validación los mejores parámetros obtenidos para regresión logística, Random Forest, Extra Trees y Balanced Random Forest. Compararemos Average Precision, ROC-AUC, precision, recall y F1 a umbral 0,5, además del tiempo de ajuste.

**Por qué importa en RetailNova:** una comparación bajo el mismo split y el mismo umbral facilita valorar la recuperación de abandonos, el volumen de contactos y el costo computacional antes de elegir un modelo.

In [6]:
supervised_results = evaluate_supervised_models(optuna_results, prepared)
display(supervised_results)
print(f"Mejor por Average Precision: {supervised_results.iloc[0]['modelo']}")

                 modelo  ...                                         parametros
0    LogisticRegression  ...  {"C": 1.3311216080736887, "class_weight": "bal...
1            ExtraTrees  ...  {"n_estimators": 140, "max_depth": 10, "min_sa...
2  BalancedRandomForest  ...  {"n_estimators": 140, "max_depth": null, "min_...
3          RandomForest  ...  {"n_estimators": 100, "max_depth": 10, "min_sa...

[4 rows x 8 columns]
Mejor por Average Precision: LogisticRegression


### Conclusión de la sección 5

La logística optimizada obtuvo la mayor Average Precision (0,4103), seguida por Extra Trees (0,4053); a umbral 0,5 la logística alcanzó recall 0,592 y F1 0,440. La selección final ajusta el umbral usando validación. Test sigue sin cargarse.

## 6. Elección del modelo ganador

**Qué haremos:** confirmaremos como ganador el modelo con mayor Average Precision en validación y fijaremos un umbral maximizando F1 en esa validación.

**Por qué importa en RetailNova:** Average Precision prioriza la calidad del ranking frente al desbalance; el umbral determina cuántos clientes se priorizan para acciones de retención, por lo que se fija antes de abrir test.

In [7]:
selection = select_model(optuna_results, prepared)
display(selection)
print(
    f"Ganador: {selection['algoritmo']}; "
    f"umbral definido en validación: {selection['umbral']:.3f}; "
    f"Average Precision: {selection['average_precision_validacion']:.4f}."
)

{'algoritmo': 'LogisticRegression', 'parametros': {'C': 1.3311216080736887, 'class_weight': 'balanced'}, 'umbral': 0.45347980171005964, 'average_precision_validacion': 0.41029212816705896, 'f1_validacion': 0.44305758420749786}
Ganador: LogisticRegression; umbral definido en validación: 0.453; Average Precision: 0.4103.


### Conclusión de la sección 6

Se eligió LogisticRegression con pesos de clase y umbral 0,453, determinados exclusivamente con train/validación. En la etapa siguiente el holdout se cargará por primera vez para la evaluación final.

## 7. Evaluación final en test y explicabilidad

**Qué haremos:** abriremos el test reservado, reajustaremos el pipeline en todo train y evaluaremos una sola vez con matriz de confusión, curvas ROC y precision–recall, lift, ganancia acumulada y una explicación SHAP sobre una muestra de clientes.

**Por qué importa en RetailNova:** las métricas de ranking permiten priorizar contactos y la matriz cuantifica aciertos y errores a un umbral operativo; SHAP ayuda a revisar qué señales empujan las predicciones, sin atribuir causalidad.

In [8]:
test_df = load_test_data()
final_metrics, scored_test = evaluate_final_test(train_df, test_df, selection, prepared)
display(final_metrics)
display(scored_test.head())
print("Curvas ROC y precision–recall: outputs/graphs/evaluacion_curvas_test.png")
print("Matriz de confusión: outputs/graphs/matriz_confusion_modelo_test.png")
print("Lift y ganancia: outputs/graphs/ganancia_lift_test.png")
print("Explicabilidad SHAP: outputs/graphs/explicabilidad_shap_test.png")
print("Modelo y métricas: outputs/model/")
print("Predicciones de test: data/processed/test/predicciones_modelo_final_test.csv")

{'modelo_seleccionado': 'LogisticRegression', 'parametros': {'C': 1.3311216080736887, 'class_weight': 'balanced'}, 'umbral_elegido_en_validacion': 0.45347980171005964, 'average_precision_validacion': 0.41029212816705896, 'f1_validacion': 0.44305758420749786, 'train_completo_refit': 148000, 'test': 37000, 'accuracy': 0.5792162162162162, 'precision': 0.32414588136658984, 'recall': 0.6984306198487072, 'f1': 0.4427901649905157, 'roc_auc': 0.6748115564299106, 'average_precision': 0.41710939594604257, 'matriz_confusion_tn_fp_fn_tp': [[15245, 12898], [2671, 6186]], 'advertencia': 'Evaluación sobre datos sintéticos; no acredita desempeño comercial ni validez en clientes reales.'}
   id_cliente         region  ...  prediccion  nivel_riesgo
0       25311  Metropolitana  ...           1          Alto
1      103301  Metropolitana  ...           1          Alto
2       33514         Biobío  ...           0         Medio
3      154171  Metropolitana  ...           0         Medio
4       62433      

### Conclusión de la sección 7

En este holdout sintético, el modelo obtuvo ROC-AUC 0,675, Average Precision 0,417, precision 0,324 y recall 0,698 al umbral fijado en validación. La matriz, curvas ROC y precision–recall, lift, ganancia y SHAP describen ranking y errores, no causalidad ni eficacia comercial. Se necesitan datos autorizados, validación temporal y costos/capacidad de contacto antes de decidir campañas.

## 8. Cierre del modelado

El flujo mantiene la separación entre segmentación no supervisada y predicción supervisada, evita sobremuestrear por defecto y reserva test para la evaluación final. Los artefactos dejan trazabilidad para la futura integración con el dashboard y la pipeline; los resultados sintéticos no son una promesa de desempeño en producción.